# Local Gaia DR3 binary stars — 3D extract

Download binary stars from Gaia DR3, compute galactocentric 3D coordinates, and export to CSV.

**Source:** Gaia DR3, table `gaiadr3.gaia_source`

**Selection criteria:**
- `non_single_star = 3` — astrometric and spectroscopic binary (NSS bit flags 1+2)
- `parallax_over_error > 5` — reliable parallax signal-to-noise
- `parallax > 0.01` mas — within ~100 kpc

**Expected volume:** ~58,209 binary stars (validated in `explore_gaia_archive.ipynb`)

**3D coordinates:** Galactocentric frame (IAU 2019) via `add_galactocentric_coords` in `helpers/coordinates.py`

**Source licence:** [CC BY-NC 3.0 IGO](https://www.cosmos.esa.int/web/gaia-users/license) — Credit: ESA, Gaia DPAC

In [1]:
from pathlib import Path

import numpy as np
from astroquery.gaia import Gaia

from helpers import add_galactocentric_coords

Gaia.MAIN_GAIA_TABLE = "gaiadr3.gaia_source"
OUTPUT_PATH = Path("data/local_galaxies_dr3.csv")
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

## Download from Gaia Archive

In [2]:
query = """
SELECT
    source_id,
    designation,
    ra,
    dec,
    parallax,
    parallax_error,
    parallax_over_error,
    distance_gspphot,
    phot_g_mean_mag
FROM gaiadr3.gaia_source
WHERE non_single_star = 3
  AND parallax_over_error > 5
  AND parallax > 0.01
"""

job = Gaia.launch_job_async(query)
table = job.get_results()

print(f"Downloaded {len(table):,} binary stars")
table.pprint(max_lines=5, max_width=130)

INFO: Query finished. [astroquery.utils.tap.core]
Downloaded 58,209 galaxies
     source_id              designation                  ra         ... parallax_over_error distance_gspphot phot_g_mean_mag
                                                        deg         ...                            pc              mag      
------------------- ---------------------------- ------------------ ... ------------------- ---------------- ---------------
 143062925384952832  Gaia DR3 143062925384952832  43.37826333973297 ...            68.88703         133.1131       12.124441
                ...                          ...                ... ...                 ...              ...             ...
6774346161003933568 Gaia DR3 6774346161003933568   315.521270863589 ...            16.58263               --       11.963295
6775301735389873664 Gaia DR3 6775301735389873664 315.75906797059713 ...            23.76069         207.3692       11.767903
Length = 58209 rows


## Compute galactocentric 3D coordinates

In [4]:
with_coords = add_galactocentric_coords(
    table,
    min_parallax_over_error=5.0,
    prefer_distance_gspphot=True,
)

parallax = np.asarray(with_coords["parallax"], dtype=float)
distance_gspphot = np.asarray(with_coords["distance_gspphot"], dtype=float)

use_gspphot = np.isfinite(distance_gspphot) & (distance_gspphot > 0)
distance_pc = np.where(use_gspphot, distance_gspphot, np.where(parallax > 0, 1000.0 / parallax, np.nan))
distance_method = np.where(use_gspphot, "distance_gspphot", "parallax")

with_coords["distance_pc"] = distance_pc
with_coords["distance_method"] = distance_method

FINAL_COLUMNS = [
    "source_id",
    "designation",
    "ra",
    "dec",
    "parallax",
    "parallax_error",
    "parallax_over_error",
    "distance_pc",
    "distance_method",
    "phot_g_mean_mag",
    "x_gc",
    "y_gc",
    "z_gc",
    "r_cyl_gc",
    "r_sph_gc",
]

export_table = with_coords[FINAL_COLUMNS].copy()

## Validation

In [5]:
n_rows = len(export_table)
n_valid_coords = np.isfinite(export_table["x_gc"]).sum()
n_gspphot = (export_table["distance_method"] == "distance_gspphot").sum()
n_parallax = (export_table["distance_method"] == "parallax").sum()

print(f"Rows: {n_rows:,}")
print(f"Rows with valid 3D coordinates: {n_valid_coords:,} ({n_valid_coords / n_rows:.1%})")
print(f"Distance from distance_gspphot: {n_gspphot:,}")
print(f"Distance from parallax: {n_parallax:,}")
print(
    f"Spherical radius (kpc): min={np.nanmin(export_table['r_sph_gc']):.3f}, "
    f"max={np.nanmax(export_table['r_sph_gc']):.3f}"
)

export_table[:5].pprint(max_lines=12, max_width=130)

Rows: 58,209
Rows with valid 3D coordinates: 36,375 (62.5%)
Distance from distance_gspphot: 36,375
Distance from parallax: 21,834
Spherical radius (kpc): min=2.526, max=17.640
    source_id              designation                 ra        ...          z_gc              r_cyl_gc          r_sph_gc    
                                                      deg        ...                                                          
------------------ --------------------------- ----------------- ... --------------------- ----------------- -----------------
143062925384952832 Gaia DR3 143062925384952832 43.37826333973297 ... -0.022446316977824624 8.407112062588714  8.40714202747108
206442963204222464 Gaia DR3 206442963204222464 73.27670873429663 ...   0.02981474369277094 8.735838945746618 8.735889823307346
209161024667432064 Gaia DR3 209161024667432064 81.07524731574468 ...   0.05113453063140953 8.578642668419365 8.578795065313557
210556034340368000 Gaia DR3 210556034340368000 84.450007328163

## Export CSV

In [6]:
export_table["source_id"] = export_table["source_id"].astype(str)
export_table.write(OUTPUT_PATH, format="ascii.csv", overwrite=True)

print(f"Exported {len(export_table):,} rows to {OUTPUT_PATH}")
print(f"File size: {OUTPUT_PATH.stat().st_size / 1e6:.1f} MB")

Exported 58,209 rows to data/local_galaxies_dr3.csv
File size: 15.2 MB


## Next steps

- Publish on [data.gouv.fr](https://www.data.gouv.fr) with Gaia attribution (Credit: ESA, Gaia DPAC)
- Use a licence compatible with CC BY-NC 3.0 IGO (non-commercial)
- Optionally add a Parquet export for programmatic reuse